## Setup

In [1]:
from __future__ import annotations

import os
import time
from pynput import keyboard

from core.config import DEFAULT_TTS_LANGUAGE, get_paths
from core.memory import MesmerlaMemory
from core.tts import load_tts_engine, load_stream

import numpy as np
import soundfile as sf



In [ ]:
# Settings
personality = "Mesmerla"
mode = "reflective"
tts_language = DEFAULT_TTS_LANGUAGE

memory = MesmerlaMemory(personality)
memory.reset()
ref_audio_path, ref_text_path, _, _, model_path = get_paths(personality, combined_audio=False)

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.


In [ ]:
coqui_engine = load_tts_engine(
    engine_type="pocket",
    ref_audio_path=ref_audio_path,
    language="en",
)

🔊 Loading RealtimeTTS pocket on cpu...
📁 pocket model directory: C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\xtts\v2.0.2


In [4]:
import logging

tts_stream = load_stream(engine=coqui_engine)

# Suppress DEBUG and INFO messages globally.
root_logger = logging.getLogger()
root_logger.setLevel(logging.WARNING)

for handler in root_logger.handlers:
    handler.setLevel(logging.WARNING)

[2026-08-16 19:13:41,873] INFO:root: loaded engine pocket_tts


In [6]:
coqui_engine.shutdown()

In [5]:
tts_stream.feed("Testing, one, two, three. This is a test of the RealtimeTTS streaming engine.")
tts_stream.play()

In [6]:
from conversation import conversation_with_AI
from core.llm import load_model
llm = load_model(
    model_path,
    n_ctx=2048,
    n_threads=os.cpu_count(),
    n_batch=64,
    verbose=False,
)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


✅ Model loaded in 40.96s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Meta-Llama-3-8B-Instruct-Q4_K_M.gguf


## Converse

In [10]:
response = conversation_with_AI(llm, coqui_engine=coqui_engine, personality="Mesmerla", mode="reflective", verbose=True, tts_language="en")

🎙️ Starting conversation...

🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav
✅ Audio recorded.
🗣️ You said: although the thing is I feel like your voice is still not great I mean I did pick pocket TTS for your voice which is I probably a lighter one and so less better quality but I hope that with more training with more training date or sample set I mean it would improve what you think about it

📂 Loaded memory from memory_logs\mesmerla_memory_Mesmerla.json
💬 Mesmerla: ⏱️ Time to first text: 1.99s
I appreciate your honesty. I can understand why you might think my voice isn't great, and to be honest, I'm still getting used to it myself. It's not easy being a shy person trying to express myself through words. But I do appreciate the effort you put into selecting a lighter TTS voice for me.

I'm glad you 

In [ ]:
from pynput import keyboard
import time
# Global control
continue_conversation = True

# Define keypress handling
def on_key_press(key):
    global continue_conversation
    if hasattr(key, 'char') and key.char == 'q':
        continue_conversation = False
        print("🛑 Stopping conversation loop... Pressed 'q'")
        return False  # Stops listener

print("🔁 Press 'q' at any time to stop.")
listener = keyboard.Listener(on_press=on_key_press)
listener.start()

try:
    while continue_conversation:
        conversation_with_AI(llm, personality="Mesmerla", mode="concise", verbose=False, tts_language="en")
        print("⏳ Listening again...")
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping conversation loop... (KeyboardInterrupt)")
    continue_conversation = False

listener.join()

## Work testing

In [ ]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()

In [ ]:
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

In [8]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.


In [ ]:
stop_mesmerla_server()

## Finetuning work

In [ ]:
from pathlib import Path
import json
import textwrap

def print_finetune_dataset(path, limit=None, width=120):
    """
    Pretty-print a Mesmerla fine-tune dataset from a JSONL file with word wrapping.
    
    Parameters:
        path (str): Path to the .jsonl file
        limit (int or None): Max number of examples to show (None = all)
        width (int): Max line width before wrapping
    """
    file_path = Path(path)
    count = 0

    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            example = json.loads(line)
            print(f"🔹 Example {i}")
            print(textwrap.fill(example["prompt"], width=width))
            print(f"💬 {textwrap.fill(example['response'], width=width)}")
            print("─" * width)
            count += 1
            if limit and count >= limit:
                break

In [ ]:
print_finetune_dataset("finetuning/mesmerla_finetune_set_batch10.jsonl")

In [ ]:
from pathlib import Path

# Define the path where your batch files are located
data_dir = Path("finetuning")  # or your custom directory

# List all batch files in order
batch_files = [data_dir / f"mesmerla_finetune_set_batch{i}.jsonl" for i in range(1, 11)]

# Output file
output_file = data_dir / "mesmerla_dataset.jsonl"

# Combine them
with output_file.open("w", encoding="utf-8") as outfile:
    for file in batch_files:
        with file.open("r", encoding="utf-8") as infile:
            lines = infile.readlines()
            outfile.writelines(lines)

print(f"✅ Merged {len(batch_files)} batches into {output_file.name}")


## conversing by chat

In [1]:
from core.config import get_paths
from core.llm import load_model
from text_convo import text_conversation
from core.memory import MesmerlaMemory

In [2]:
# Load the model path dynamically
_, _, _, _, model_path = get_paths("HuTao")  # Or "HuTao", "Zhongli"
llm = load_model(model_path, verbose=False)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


✅ Model loaded in 29.60s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Meta-Llama-3-8B-Instruct-Q4_K_M.gguf


In [3]:

user_message = """Hi How are you ?"""

# Get reply
reply = text_conversation(llm, user_message, personality="Mesmerla", mode="reflective", verbose=False, jupyter_notebook=True)

#print("\n", prompt)

In [ ]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

In [ ]:
memory.reset()